# 11 · Ejercicios de pandas tipo entrevista (con solución)

Diez ejercicios cortos con las tablas Gold de la plataforma. Cubren lo que suelen preguntar en una prueba técnica de analista de datos: leer y explorar, nulos, duplicados, filtrar y ordenar, fechas, `merge`, `pivot_table`, `apply` con una función propia, `groupby` con `transform` y un caso final que combina todo.

**Cómo practicar**

1. Lee el enunciado y la pista.
2. Escribe tu respuesta en la celda **✏️ Tu intento** (borra el `#` de las líneas que vayas usando).
3. Antes de mirar la solución, **explícalo en voz alta en 3 pasos**: qué datos uso, qué operación aplico y qué resultado espero.
4. Ejecuta la celda **✅ Solución** y compara.

**Cómo correrlo:** conecta el notebook a **Serverless** y ejecuta primero la celda de *Preparación*. Las celdas de intento no rompen el *Run all* aunque las dejes vacías.

| # | Tema | Funciones clave |
|---|---|---|
| 1 | Leer y explorar | `shape`, `info`, `describe`, `nunique`, `pd.to_datetime` |
| 2 | Nulos | `isna().sum()`, `dropna`, por qué no usar `fillna(0)` a ciegas |
| 3 | Duplicados | `duplicated(subset=…)`, `drop_duplicates(keep=…)` |
| 4 | Filtrar y ordenar | `query`, `nlargest`, `sort_values` |
| 5 | Fechas | `.dt.year`, `.dt.day_name()`, `resample("MS")` |
| 6 | Unir tablas | `merge(how=…, validate=…, indicator=True)` |
| 7 | Tabla dinámica | `pivot_table(index, columns, values, aggfunc, margins)` |
| 8 | Función propia | `def`, `apply`, `np.select`, `pd.crosstab` |
| 9 | Comparar dentro de cada grupo | `groupby().transform()` vs `agg()` |
| 10 | Caso combinado | Todo lo anterior: costo de un tanque de gasolina en dólares |

## Preparación

La función `leer()` carga una tabla Gold y convierte las columnas `DECIMAL` a `double`, para que pandas las trate como números normales. Siempre se filtra y se eligen columnas **en Spark, antes de `toPandas()`**: así solo viaja a memoria lo necesario.

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

GOLD = "rd_economic_intelligence.gold"
PRINCIPALES = ["gasolina_premium", "gasolina_regular", "gasoil_optimo", "gasoil_regular", "glp", "kerosene"]

def leer(tabla):
    """Carga una tabla Gold y convierte DECIMAL a double para trabajar cómodo en pandas."""
    df = spark.table(f"{GOLD}.{tabla}")
    for columna, tipo in df.dtypes:
        if tipo.startswith("decimal"):
            df = df.withColumn(columna, F.col(columna).cast("double"))
    return df

print("Listo")

## Ejercicio 1 · Leer y explorar una tabla

**Enunciado:** carga `fuel_price_metrics` con las columnas `fuel_code`, `fuel_name`, `period_start_date`, `price_dop` y `weekly_change_pct`. Responde: ¿cuántas filas y columnas tiene?, ¿de qué tipo es cada columna?, ¿cuántos combustibles distintos hay? y ¿qué rango de fechas cubre?

**Pista:** `shape`, `info()`, `describe()`, `nunique()`. Fíjate en el tipo de `period_start_date`: llega como `object` (fechas de Python), no como fecha de pandas. Conviértela con `pd.to_datetime`.

**Por qué lo preguntan:** es lo primero que se hace con cualquier dato nuevo. Muestra que no empiezas a calcular sin conocer la tabla.

In [0]:
# fuel = leer("fuel_price_metrics").select(...).toPandas()
# print(fuel.shape)

In [0]:
fuel = (leer("fuel_price_metrics")
        .select("fuel_code", "fuel_name", "period_start_date", "price_dop", "weekly_change_pct")
        .toPandas())

print("Filas y columnas:", fuel.shape)
print("Tipo de period_start_date antes:", fuel["period_start_date"].dtype)
fuel["period_start_date"] = pd.to_datetime(fuel["period_start_date"])   # object -> datetime64
print("Tipo de period_start_date después:", fuel["period_start_date"].dtype)

fuel.info()
print("Combustibles distintos:", fuel["fuel_code"].nunique())
print("Desde", fuel["period_start_date"].min().date(), "hasta", fuel["period_start_date"].max().date())
display(fuel.describe().round(2).reset_index())

## Ejercicio 2 · Nulos: encontrarlos y decidir qué hacer

**Enunciado:** carga la inflación general (`inflation_metrics` con `series_type = 'headline'`) con `reference_month`, `index_value`, `mom_inflation_pct` y `yoy_inflation_pct`. Cuenta los nulos por columna y su porcentaje. ¿En qué meses falta la inflación interanual y por qué?

**Pista:** `isna().sum()`. Divide entre `len(df)` para el porcentaje.

**Lo importante para la entrevista:** un nulo no siempre es un error. La inflación interanual compara contra el mismo mes del año anterior, así que los primeros 12 meses de la serie **no pueden** tener ese valor. Rellenarlos con `fillna(0)` diría que la inflación fue 0 %, que es falso. Lo correcto aquí es dejarlos nulos o filtrarlos con `dropna(subset=[...])` cuando el cálculo lo necesite.

In [0]:
# infl = leer("inflation_metrics").filter("series_type = 'headline'").select(...).toPandas()
# infl.isna().sum()

In [0]:
infl = (leer("inflation_metrics")
        .filter("series_type = 'headline'")
        .select("reference_month", "index_value", "mom_inflation_pct", "yoy_inflation_pct")
        .toPandas())
infl["reference_month"] = pd.to_datetime(infl["reference_month"])

nulos = infl.isna().sum().to_frame("nulos")
nulos["porcentaje"] = (nulos["nulos"] / len(infl) * 100).round(1)
display(nulos.rename_axis("columna").reset_index())

sin_yoy = infl.loc[infl["yoy_inflation_pct"].isna(), "reference_month"].sort_values()
print(f"Meses sin inflación interanual: {len(sin_yoy)}, del {sin_yoy.min():%Y-%m} al {sin_yoy.max():%Y-%m}")
print("Primer mes de la serie:", f"{infl['reference_month'].min():%Y-%m}")

# Para calcular con la columna, se filtran los nulos (no se rellenan con 0):
con_yoy = infl.dropna(subset=["yoy_inflation_pct"])
print("Filas útiles para análisis interanual:", len(con_yoy))

## Ejercicio 3 · Duplicados

**Enunciado:** la clave de la tabla de combustibles es `fuel_code` + `period_start_date` (un precio por combustible por semana). Simula una carga repetida: agrega otra vez las últimas 5 semanas de gasolina regular. Luego encuentra los duplicados por la clave, muéstralos y elimínalos quedándote con la última versión.

**Pista:** `pd.concat`, `duplicated(subset=clave)`, `keep=False` para ver todas las copias, `drop_duplicates(subset=clave, keep="last")`.

**Lo importante:** revisar duplicados **por la clave del negocio**, no por la fila completa. Dos filas pueden tener la misma clave y un precio distinto (una corrección); `duplicated()` sin `subset` no las detectaría.

In [0]:
# clave = ["fuel_code", "period_start_date"]
# con_dup = pd.concat([fuel, ...], ignore_index=True)

In [0]:
clave = ["fuel_code", "period_start_date"]
print("Duplicados en la tabla real:", fuel.duplicated(subset=clave).sum())

repetidas = fuel[fuel["fuel_code"] == "gasolina_regular"].sort_values("period_start_date").tail(5)
con_dup = pd.concat([fuel, repetidas], ignore_index=True)
print("Filas antes:", len(fuel), "| después de la carga repetida:", len(con_dup))
print("Filas duplicadas por clave:", con_dup.duplicated(subset=clave).sum())

display(con_dup[con_dup.duplicated(subset=clave, keep=False)].sort_values(clave))

limpio = con_dup.drop_duplicates(subset=clave, keep="last")
print("Filas después de limpiar:", len(limpio))
assert len(limpio) == len(fuel.drop_duplicates(subset=clave))

## Ejercicio 4 · Filtrar y ordenar

**Enunciado:** desde 2020, ¿cuáles fueron las 5 semanas con la mayor subida semanal (%) de la **gasolina premium**? Muestra fecha, precio y variación.

**Pista:** `query()` para filtrar con texto legible y `nlargest(5, columna)` para el top. Es igual a `sort_values(ascending=False).head(5)`.

In [0]:
# premium = fuel.query("fuel_code == 'gasolina_premium' and period_start_date >= '2020-01-01'")
# premium.nlargest(...)

In [0]:
premium = fuel.query("fuel_code == 'gasolina_premium' and period_start_date >= '2020-01-01'")

top5 = premium.nlargest(5, "weekly_change_pct")[["period_start_date", "price_dop", "weekly_change_pct"]]
display(top5)

# La misma respuesta con sort_values:
otra_forma = premium.sort_values("weekly_change_pct", ascending=False).head(5)
assert top5.index.tolist() == otra_forma.index.tolist()

## Ejercicio 5 · Fechas

**Enunciado:** con la tasa de **venta** del dólar diaria (`exchange_rate_metrics`, `rate_type = 'sell'`):

1. Agrega columnas de año, mes y nombre del día.
2. Comprueba qué días de la semana tienen datos. ¿Hay fines de semana? Una tabla de tasas no siempre trae solo días hábiles: verifícalo en vez de suponerlo.
3. Calcula el promedio mensual desde enero de 2025.
4. Di cuál es la última fecha disponible.

**Pista:** accesor `.dt` (`.dt.year`, `.dt.month`, `.dt.day_name()`). Para el promedio mensual: `set_index("rate_date")["rate"].resample("MS").mean()` (`MS` = inicio de mes).

In [0]:
# fx = leer("exchange_rate_metrics").filter("rate_type = 'sell'").select("rate_date", "rate").toPandas()
# fx["rate_date"] = pd.to_datetime(fx["rate_date"])

In [0]:
fx = leer("exchange_rate_metrics").filter("rate_type = 'sell'").select("rate_date", "rate").toPandas()
fx["rate_date"] = pd.to_datetime(fx["rate_date"])

fx["anio"] = fx["rate_date"].dt.year
fx["mes"] = fx["rate_date"].dt.month
fx["dia_semana"] = fx["rate_date"].dt.day_name()

print("Días con datos (casi todo de lunes a viernes; hay algunos sábados y domingos publicados por el BCRD):")
print(fx["dia_semana"].value_counts())

mensual = fx.set_index("rate_date")["rate"].resample("MS").mean().round(4)
display(mensual.loc["2025-01-01":].rename("tasa_promedio").reset_index())

print("Última fecha disponible:", fx["rate_date"].max().date())

## Ejercicio 6 · Unir dos tablas con `merge`

**Enunciado:** une la tasa promedio mensual del dólar (`exchange_rate_monthly_metrics`, venta, columnas `month_start` y `month_avg_rate`) con la inflación interanual del ejercicio 2, por mes. Usa un `left` join desde el dólar. ¿Qué meses del dólar no tienen inflación y por qué? ¿Qué cambia con `how="inner"`?

**Pista:** renombra las columnas de fecha a un nombre común (`mes`). `validate="one_to_one"` hace que pandas falle si la clave se repite, y `indicator=True` agrega la columna `_merge` que dice de qué lado vino cada fila.

**Equivalencia con SQL:** `how="left"` es `LEFT JOIN`, `how="inner"` es `INNER JOIN`, `how="outer"` es `FULL OUTER JOIN`.

In [0]:
# dolar_m = leer("exchange_rate_monthly_metrics").filter("rate_type = 'sell'").select(...).toPandas()
# unido = dolar_m.merge(infl_m, on="mes", how="left", validate="one_to_one", indicator=True)

In [0]:
dolar_m = (leer("exchange_rate_monthly_metrics")
           .filter("rate_type = 'sell'")
           .select(F.col("month_start").alias("mes"), F.col("month_avg_rate").alias("dolar_promedio"))
           .toPandas())
dolar_m["mes"] = pd.to_datetime(dolar_m["mes"])
infl_m = infl[["reference_month", "yoy_inflation_pct"]].rename(columns={"reference_month": "mes"})

unido = dolar_m.merge(infl_m, on="mes", how="left", validate="one_to_one", indicator=True)
print(unido["_merge"].value_counts())
print("Meses del dólar sin inflación:", unido.loc[unido["_merge"] == "left_only", "mes"].dt.strftime("%Y-%m").tolist())

inner = dolar_m.merge(infl_m, on="mes", how="inner")
print("Filas con left:", len(unido), "| con inner:", len(inner))
display(unido.drop(columns="_merge").query("mes >= '2025-07-01'").round(2))

**Lectura:** el dólar se publica a diario y ya tiene el mes en curso; el IPC sale con un mes de rezago. Por eso el último mes del dólar queda `left_only`. Con `inner` esos meses desaparecen. Elegir el tipo de join es una decisión de negocio: ¿quiero ver todos los meses del dólar aunque falte la inflación, o solo los meses completos?

## Ejercicio 7 · Tabla dinámica con `pivot_table`

**Enunciado:** con `electricity_metrics`, arma una tabla con los **años en las filas**, las **distribuidoras en las columnas** y el **promedio de pérdidas (%)** en las celdas. Después, arma otra con la **energía comprada (GWh) sumada por año** y una fila y columna de totales.

**Pista:** `pivot_table(index="anio", columns="distributor", values=..., aggfunc="mean")`. Para totales: `margins=True, margins_name="Total"`.

**Trampa:** la tabla trae una fila `Total EDE` que ya es la suma de las tres distribuidoras. Si la dejas al sumar energía, el total sale el doble. Quítala antes.

In [0]:
# elec = leer("electricity_metrics").select("reference_month", "distributor", "losses_pct", "energy_bought_gwh").toPandas()
# elec.pivot_table(...)

In [0]:
elec = leer("electricity_metrics").select("reference_month", "distributor", "losses_pct", "energy_bought_gwh").toPandas()
elec["anio"] = pd.to_datetime(elec["reference_month"]).dt.year

perdidas = elec.pivot_table(index="anio", columns="distributor", values="losses_pct", aggfunc="mean").round(2)
display(perdidas.reset_index())

solo_ede = elec[elec["distributor"] != "Total EDE"]       # evitar contar dos veces
energia = solo_ede.pivot_table(index="anio", columns="distributor", values="energy_bought_gwh",
                               aggfunc="sum", margins=True, margins_name="Total").round(1)
energia.index = energia.index.astype(str)   # la fila "Total" es texto: el índice no puede mezclar números y texto al mostrarlo
display(energia.reset_index())

**Diferencia con `pivot()`:** `pivot()` solo reacomoda y falla si hay dos filas con la misma combinación de fila y columna. `pivot_table()` agrega (promedio, suma…), así que funciona con varios meses por año. La operación inversa, de ancho a largo, es `melt()`.

## Ejercicio 8 · Función propia con `apply`

**Enunciado:** con los precios agrícolas al detalle (`agri_price_monthly`, `channel = 'retail'`, `market = 'Promedio mercados'`), toma el **último mes** y clasifica cada producto según su variación interanual (`yoy_pct`):

| Regla | Etiqueta |
|---|---|
| ≥ 10 % | Subió fuerte |
| > 2 % y < 10 % | Subió |
| entre −2 % y 2 % | Estable |
| < −2 % | Bajó |

Cuenta cuántos productos hay en cada grupo y crúzalo con el rubro.

**Pista:** escribe `def clasificar(v):` con `if/elif` y aplícala con `df["yoy_pct"].apply(clasificar)`. Para el cruce: `pd.crosstab(df["rubro"], df["tendencia"])`.

**Punto extra:** `apply` llama a la función fila por fila. Con muchos datos es más rápido `np.select`, que evalúa toda la columna de una vez (vectorizado). La solución hace las dos y comprueba que coinciden.

In [0]:
# def clasificar(v):
#     if v >= 10:
#         return "Subió fuerte"
#     ...

In [0]:
agri = (leer("agri_price_monthly")
        .filter("channel = 'retail' AND market = 'Promedio mercados'")
        .select("reference_month", "rubro", "product", "yoy_pct")
        .toPandas())
ultimo = agri["reference_month"].max()
a = agri[(agri["reference_month"] == ultimo) & agri["yoy_pct"].notna()].copy()
print("Mes analizado:", ultimo, "| productos con variación interanual:", len(a))

def clasificar(v):
    """Etiqueta una variación interanual en %."""
    if v >= 10:
        return "1 · Subió fuerte"
    elif v > 2:
        return "2 · Subió"
    elif v >= -2:
        return "3 · Estable"
    return "4 · Bajó"

a["tendencia"] = a["yoy_pct"].apply(clasificar)
display(a["tendencia"].value_counts().sort_index().rename_axis("tendencia").reset_index(name="productos"))
display(pd.crosstab(a["rubro"], a["tendencia"], margins=True, margins_name="Total").reset_index())

# Versión vectorizada
a["tendencia_np"] = np.select([a["yoy_pct"] >= 10, a["yoy_pct"] > 2, a["yoy_pct"] >= -2],
                              ["1 · Subió fuerte", "2 · Subió", "3 · Estable"], default="4 · Bajó")
assert (a["tendencia"] == a["tendencia_np"]).all()
print("apply y np.select dan el mismo resultado")

## Ejercicio 9 · Comparar cada fila con su grupo (`transform`)

**Enunciado:** para los 6 combustibles principales en 2026, agrega a cada semana el **precio promedio de ese combustible en 2026** y la diferencia en % contra ese promedio. Muestra la última semana de cada combustible.

**Pista:** `groupby("fuel_code")["price_dop"].transform("mean")`.

**Diferencia clave (pregunta frecuente):**
- `agg("mean")` **reduce**: devuelve una fila por grupo (6 filas).
- `transform("mean")` **conserva** todas las filas y repite el valor del grupo en cada una. Sirve para comparar cada fila con su grupo sin hacer un `merge`. Es el equivalente de `AVG(...) OVER (PARTITION BY ...)` en SQL.

In [0]:
# f26 = fuel[(fuel["period_start_date"].dt.year == 2026) & fuel["fuel_code"].isin(PRINCIPALES)].copy()
# f26["promedio_2026"] = ...

In [0]:
f26 = fuel[(fuel["period_start_date"].dt.year == 2026) & fuel["fuel_code"].isin(PRINCIPALES)].copy()

f26["promedio_2026"] = f26.groupby("fuel_code")["price_dop"].transform("mean")
f26["vs_promedio_pct"] = (f26["price_dop"] / f26["promedio_2026"] - 1) * 100

ultima = f26.sort_values("period_start_date").groupby("fuel_code").tail(1)
display(ultima[["fuel_name", "period_start_date", "price_dop", "promedio_2026", "vs_promedio_pct"]].round(2))

print("Filas con transform:", len(f26), "| filas con agg:", len(f26.groupby("fuel_code")["price_dop"].agg("mean")))

## Ejercicio 10 · Caso combinado: ¿cuánto cuesta llenar el tanque, en dólares?

**Enunciado:** un tanque de 15 galones de **gasolina regular**. Para cada mes desde enero de 2024 calcula el costo en pesos (precio promedio del mes × 15) y en dólares (costo en pesos ÷ tasa promedio del mes). ¿En qué mes fue más caro y más barato en dólares?

**Pasos que deberías poder explicar en voz alta:**
1. Filtrar gasolina regular (semanal).
2. Llevar cada semana a su mes (`.dt.to_period("M").dt.to_timestamp()`) y promediar por mes.
3. Unir con el dólar mensual del ejercicio 6 (`inner`, porque necesito los dos valores).
4. Calcular las dos columnas nuevas.
5. Encontrar el máximo y el mínimo con `idxmax()` / `idxmin()`.

In [0]:
# gas = fuel[fuel["fuel_code"] == "gasolina_regular"].copy()
# gas["mes"] = ...

In [0]:
GALONES = 15
gas = fuel[fuel["fuel_code"] == "gasolina_regular"].copy()
gas["mes"] = gas["period_start_date"].dt.to_period("M").dt.to_timestamp()
gas_m = gas.groupby("mes", as_index=False)["price_dop"].mean()

tanque = gas_m.merge(dolar_m, on="mes", how="inner", validate="one_to_one")
tanque["tanque_dop"] = tanque["price_dop"] * GALONES
tanque["tanque_usd"] = tanque["tanque_dop"] / tanque["dolar_promedio"]
tanque = tanque[tanque["mes"] >= "2024-01-01"].reset_index(drop=True)
display(tanque.round(2))

caro, barato = tanque.loc[tanque["tanque_usd"].idxmax()], tanque.loc[tanque["tanque_usd"].idxmin()]
print(f"Más caro en dólares:  {caro['mes']:%Y-%m}  US${caro['tanque_usd']:.2f}  (RD${caro['tanque_dop']:,.2f})")
print(f"Más barato en dólares: {barato['mes']:%Y-%m}  US${barato['tanque_usd']:.2f}  (RD${barato['tanque_dop']:,.2f})")

## Resumen para repasar antes de la entrevista

| Necesito… | pandas | SQL equivalente |
|---|---|---|
| Ver tamaño y tipos | `df.shape`, `df.info()` | `DESCRIBE tabla` |
| Contar nulos | `df.isna().sum()` | `SUM(CASE WHEN col IS NULL THEN 1 ELSE 0 END)` |
| Quitar duplicados por clave | `drop_duplicates(subset=…, keep="last")` | `ROW_NUMBER() OVER (PARTITION BY clave ORDER BY …) = 1` |
| Filtrar | `query("col > 5")`, `df[df.col > 5]` | `WHERE` |
| Top N | `nlargest(n, col)` | `ORDER BY col DESC LIMIT n` |
| Agrupar | `groupby().agg()` | `GROUP BY` |
| Valor del grupo en cada fila | `groupby().transform()` | `AVG() OVER (PARTITION BY …)` |
| Unir | `merge(how="left")` | `LEFT JOIN` |
| Tabla dinámica | `pivot_table()` | `PIVOT` o `SUM(CASE WHEN …)` |
| Regla propia | `apply(funcion)` / `np.select` | `CASE WHEN` |
| Agrupar por mes | `resample("MS")`, `.dt.to_period("M")` | `DATE_TRUNC('month', fecha)` |

**Frase útil:** "Filtro y resumo en Spark o SQL cuando los datos son grandes, y paso a pandas cuando ya caben en memoria para explorar, graficar o hacer reglas de negocio puntuales."